# Zero-LoRA Phase 3: Lambda Grid
Runs the initial 7-point grid to test the preservation losses.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/1forsithlord/zero-lora.git /content/zero-lora
%cd /content/zero-lora
!pip -q install transformers==5.18.0 nltk huggingface_hub pyyaml scipy matplotlib

In [ ]:
# Build reference pool (fast mode)
!python scripts/build_reference_pool.py --root /content/data --out data/reference_pool --shard-ids 0 --target-size 2000 --skip-eval-check --probe-classname-mode off

In [ ]:
# Run the grid
import subprocess
import os
import shutil

drive_dir = '/content/drive/MyDrive/zero-lora-phase3'
os.makedirs(drive_dir, exist_ok=True)

configs = [
    ["--loss", "raw", "--lambda", "0.0"],
    ["--loss", "raw", "--lambda", "0.1"],
    ["--loss", "raw", "--lambda", "1.0"],
    ["--loss", "raw", "--lambda", "10.0"],
    ["--loss", "gnsp", "--lambda", "10.0"],
    ["--loss", "gnsp", "--lambda", "100.0"],
    ["--loss", "gnsp", "--lambda", "1000.0"]
]

for cfg in configs:
    loss, lam = cfg[1], cfg[3]
    print(f"Starting: {loss} {lam}")
    cmd = ["python", "-m", "zerolora.train", "--root", "/content/data", "--task", "dtd",
           "--ref-pool", "data/reference_pool", "--epochs", "3", "--lr", "1e-4", "--seed", "0",
           "--out", "results/phase3_grid", "--tag", f"lam{lam}"] + cfg
    env = os.environ.copy()
    env["PYTHONPATH"] = "src"
    subprocess.run(cmd, env=env, check=True)
    
    if os.path.exists("results/phase3_grid"):
        for folder in os.listdir("results/phase3_grid"):
            src = os.path.join("results/phase3_grid", folder)
            dst = os.path.join(drive_dir, folder)
            if os.path.isdir(src): shutil.copytree(src, dst, dirs_exist_ok=True)